# COMP5318 Assignment 1: Rice Classification

##### Group number: 197
###### Student 1 SID: 530839244
###### Student 2 SID: 540958494
###### Student 3 SID: 550120560
###### Student 4 SID: ... 

## **1. Data Pre-processing**

In [39]:
# Import all libraries
import pandas as pd
import numpy as np

# Preprocessing 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC


# Analysis
from sklearn.model_selection import cross_val_score


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

In [40]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [41]:
# Load the rice dataset: rice-final2.csv
rice_df = pd.read_csv("rice-final2.csv")

print(rice_df.head())


    Area    Perimiter Major_Axis_Length Minor_Axis_Length Eccentricity  \
0  12573  461.4660034       192.9033508       84.57207489  0.898771763   
1  12845  464.1210022       194.3322144       85.52433777  0.897951961   
2  14055  488.7489929       207.7517548       87.25032806  0.907536149   
3  14412  490.3240051       207.4761353       89.68951416  0.901735425   
4  14658  477.1170044       189.5666351       99.99777985  0.849550545   

  Convex_Area       Extent   class  
0       12893  0.550433397  class2  
1       13125  0.774962306  class2  
2       14484  0.550076306  class1  
3       14703  0.598853171  class1  
4       15048  0.649503708  class2  


In [42]:
##### Pre-process dataset #####

## Fill in missing attribute values ##

rice_df = rice_df.replace(['?', 'NA', 'N/A', 'na', ''], np.nan)        # Replace common blanks with NaN

target_col = rice_df.columns[-1]
x = rice_df.drop(columns=[target_col])
y = rice_df[target_col]

imp_mean = SimpleImputer(missing_values = np.nan, strategy = 'mean')  # Create imputer 
x = imp_mean.fit_transform(x)       # Find means of columns and replace blank values


## Normalise the data ##
scaler = MinMaxScaler()     # Create scaler
x = scaler.fit_transform(x)  # Normalise numerical data between 0 and 1

## Change class values ##
y = y.replace({"class1": 0, "class2": 1})


In [43]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data arraCy and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])


print_data(x, y)
            


0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [44]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [45]:
# Logistic Regression

log_reg_model = LogisticRegression(random_state=0).fit(x, y) # Create and fit Model

scores = cross_val_score(log_reg_model, x, y, cv=cvKFold) # Calculate scores for each fold 

print("-- Logistic Regression --")
print(f"Scores for each fold:")
for score in scores:
    print(f"{score:.3f}")
print(f"\nAverage score: {scores.mean():.3f}")
lr_score = scores.mean()


-- Logistic Regression --
Scores for each fold:
0.914
0.936
0.964
0.943
0.950
0.929
0.943
0.950
0.893
0.964

Average score: 0.939


In [46]:
# Naïve Bayes
nb_model = GaussianNB().fit(x, y)

scores = cross_val_score(nb_model, x, y, cv=cvKFold) # Calculate scores for each fold 

print("-- Gaussian Naives Bayes --")
print(f"Scores for each fold:")
for score in scores:
    print(f"{score:.3f}")
print(f"\nAverage score: {scores.mean():.3f}")
nb_score = scores.mean()


-- Gaussian Naives Bayes --
Scores for each fold:
0.900
0.929
0.957
0.914
0.943
0.936
0.943
0.943
0.864
0.936

Average score: 0.926


### Part 1 Results


In [47]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {lr_score:.4f}")
print(f"NB average cross-validation accuracy: {nb_score:.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning

In [48]:
# Grid Search (Part 2 common setup)

# 1) Train/Test split 
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=0
)

# 2) Common runner for all classifiers in Part 2
def run_grid_search(model_name, estimator, param_grid):
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cvKFold,           # StratifiedKFold defined earlier
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    y_pred = grid.best_estimator_.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)

    print(f"-- {model_name} --")
    print(f"{model_name} best params: {grid.best_params_}")
    print(f"{model_name} cross-validation accuracy: {grid.best_score_:.4f}")
    print(f"{model_name} test set accuracy: {test_acc:.4f}")

    return grid, y_pred

In [49]:
# KNN 
# parameters you may consider

knn_param_grid = {
    "n_neighbors": [1, 3, 5, 7],
    "p": [1, 2]
}

knn_grid, knn_pred = run_grid_search("KNN", KNeighborsClassifier(), knn_param_grid)

# Print best score
print(f"\nBest Average score: {knn_grid.best_score_:.4f}")
print(f"Best k: {knn_grid.best_params_['n_neighbors']}")
print(f"Best p: {knn_grid.best_params_['p']}")

-- KNN --
KNN best params: {'n_neighbors': 7, 'p': 2}
KNN cross-validation accuracy: 0.9375
KNN test set accuracy: 0.9250

Best Average score: 0.9375
Best k: 7
Best p: 2


In [50]:
# Decision Tree 
# parameters you may consider
dt_param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid, dt_pred = run_grid_search(
    "DT",
    DecisionTreeClassifier(random_state=0),
    dt_param_grid
)

# Print best score
print(f"\nBest Average score: {dt_grid.best_score_:.4f}")
print(f"Best depth: {dt_grid.best_params_['max_depth']}")
print(f"Best min_samples_split: {dt_grid.best_params_['min_samples_split']}")
print(f"Best min_samples_leaf: {dt_grid.best_params_['min_samples_leaf']}")

-- DT --
DT best params: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
DT cross-validation accuracy: 0.9357
DT test set accuracy: 0.9429

Best Average score: 0.9357
Best depth: 3
Best min_samples_split: 2
Best min_samples_leaf: 1


In [51]:
# Ada Boost
# parameters you may consider
ada_param_grid = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.1, 0.2, 0.3, 0.5]
}

ada_grid, ada_pred = run_grid_search(
    "Ada",
    AdaBoostClassifier(random_state=0),
    ada_param_grid
)

# Print best score
print(f"\nBest Average score: {ada_grid.best_score_:.4f}")
print(f"Best n_estimators: {ada_grid.best_params_['n_estimators']}")
print(f"Best learning_rate: {ada_grid.best_params_['learning_rate']}")

-- Ada --
Ada best params: {'learning_rate': 0.2, 'n_estimators': 150}
Ada cross-validation accuracy: 0.9455
Ada test set accuracy: 0.9429

Best Average score: 0.9455
Best n_estimators: 150
Best learning_rate: 0.2


In [52]:
# Gradient Boost
# parameters you may consider
gb_param_grid = {
    "max_depth": [1, 3, 5, 7],
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.1, 0.2, 0.3, 0.5]
}

gb_grid, gb_pred = run_grid_search(
    "GB",
    GradientBoostingClassifier(random_state=0),
    gb_param_grid
)

# Print best score
print(f"\nBest Average score: {gb_grid.best_score_:.4f}")
print(f"Best max_depth: {gb_grid.best_params_['max_depth']}")
print(f"Best n_estimators: {gb_grid.best_params_['n_estimators']}")
print(f"Best learning_rate: {gb_grid.best_params_['learning_rate']}")

-- GB --
GB best params: {'learning_rate': 0.1, 'max_depth': 1, 'n_estimators': 50}
GB cross-validation accuracy: 0.9446
GB test set accuracy: 0.9429

Best Average score: 0.9446
Best max_depth: 1
Best n_estimators: 50
Best learning_rate: 0.1


In [53]:
# Random Forest
# Use information gain (entropy) and max_features='sqrt' in estimator
# parameters you may consider
rf_param_grid = {
    "n_estimators": [10, 30, 60, 100],
    "max_leaf_nodes": [6, 12]
}

rf_grid, rf_pred = run_grid_search(
    "RF",
    RandomForestClassifier(
        criterion="entropy",
        max_features="sqrt",
        random_state=0
    ),
    rf_param_grid
)

# Print best score
print(f"\nBest Average score: {rf_grid.best_score_:.4f}")
print(f"Best n_estimators: {rf_grid.best_params_['n_estimators']}")
print(f"Best max_leaf_nodes: {rf_grid.best_params_['max_leaf_nodes']}")
print(f"RF test set accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"RF test set macro average F1: {f1_score(y_test, rf_pred, average='macro'):.4f}")
print(f"RF test set weighted average F1: {f1_score(y_test, rf_pred, average='weighted'):.4f}")

-- RF --
RF best params: {'max_leaf_nodes': 6, 'n_estimators': 30}
RF cross-validation accuracy: 0.9411
RF test set accuracy: 0.9429

Best Average score: 0.9411
Best n_estimators: 30
Best max_leaf_nodes: 6
RF test set accuracy: 0.9429
RF test set macro average F1: 0.9414
RF test set weighted average F1: 0.9427


In [54]:
# SVM
# parameters you may consider
svm_param_grid = {
    "C": [0.01, 0.1, 1, 5],
    "gamma": [0.01, 0.1, 1, 10],
    "kernel": ["rbf"]   # optional in template; fix to rbf for this search
}

svm_grid, svm_pred = run_grid_search("SVM", SVC(random_state=0), svm_param_grid)

print(f"\nBest Average score: {svm_grid.best_score_:.4f}")
print(f"Best C: {svm_grid.best_params_['C']}")
print(f"Best gamma: {svm_grid.best_params_['gamma']}")
print(f"Best kernel: {svm_grid.best_params_['kernel']}")
print(f"SVM test set accuracy: {accuracy_score(y_test, svm_pred):.4f}")

-- SVM --
SVM best params: {'C': 5, 'gamma': 1, 'kernel': 'rbf'}
SVM cross-validation accuracy: 0.9429
SVM test set accuracy: 0.9321

Best Average score: 0.9429
Best C: 5
Best gamma: 1
Best kernel: rbf
SVM test set accuracy: 0.9321


### Part 2: Results

In [55]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# --- Part 2 Results ---

# KNN
knn_grid, knn_pred = run_grid_search("KNN", KNeighborsClassifier(), knn_param_grid)
print(f"KNN best k: {knn_grid.best_params_['n_neighbors']}")
print(f"KNN best p: {knn_grid.best_params_['p']}")
print(f"KNN cross-validation accuracy: {knn_grid.best_score_:.4f}")
print(f"KNN test set accuracy: {accuracy_score(y_test, knn_pred):.4f}")
print()

# DT
dt_grid, dt_pred = run_grid_search("DT", DecisionTreeClassifier(random_state=0), dt_param_grid)
print(f"DT best max_depth: {dt_grid.best_params_['max_depth']}")
print(f"DT best min_samples_split: {dt_grid.best_params_['min_samples_split']}")
print(f"DT best min_samples_leaf: {dt_grid.best_params_['min_samples_leaf']}")
print(f"DT cross-validation accuracy: {dt_grid.best_score_:.4f}")
print(f"DT test set accuracy: {accuracy_score(y_test, dt_pred):.4f}")
print()

# Ada
ada_grid, ada_pred = run_grid_search("Ada", AdaBoostClassifier(random_state=0), ada_param_grid)
print(f"Ada best n_estimators: {ada_grid.best_params_['n_estimators']}")
print(f"Ada best learning_rate: {ada_grid.best_params_['learning_rate']}")
print(f"Ada cross-validation accuracy: {ada_grid.best_score_:.4f}")
print(f"Ada test set accuracy: {accuracy_score(y_test, ada_pred):.4f}")
print()

# GB
gb_grid, gb_pred = run_grid_search("GB", GradientBoostingClassifier(random_state=0), gb_param_grid)
print(f"GB best max_depth: {gb_grid.best_params_['max_depth']}")
print(f"GB best n_estimators: {gb_grid.best_params_['n_estimators']}")
print(f"GB best learning_rate: {gb_grid.best_params_['learning_rate']}")
print(f"GB cross-validation accuracy: {gb_grid.best_score_:.4f}")
print(f"GB test set accuracy: {accuracy_score(y_test, gb_pred):.4f}")
print()

# RF
print(f"RF best n_estimators: {rf_grid.best_params_['n_estimators']}")
print(f"RF best max_leaf_nodes: {rf_grid.best_params_['max_leaf_nodes']}")
print(f"RF cross-validation accuracy: {rf_grid.best_score_:.4f}")
print(f"RF test set accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"RF test set macro average F1: {f1_score(y_test, rf_pred, average='macro'):.4f}")
print(f"RF test set weighted average F1: {f1_score(y_test, rf_pred, average='weighted'):.4f}")
print()

# SVM
print(f"SVM best C: {svm_grid.best_params_['C']}")
print(f"SVM best gamma: {svm_grid.best_params_['gamma']}")
print(f"SVM best kernel: {svm_grid.best_params_['kernel']}")
print(f"SVM cross-validation accuracy: {svm_grid.best_score_:.4f}")
print(f"SVM test set accuracy: {accuracy_score(y_test, svm_pred):.4f}")



-- KNN --
KNN best params: {'n_neighbors': 7, 'p': 2}
KNN cross-validation accuracy: 0.9375
KNN test set accuracy: 0.9250
KNN best k: 7
KNN best p: 2
KNN cross-validation accuracy: 0.9375
KNN test set accuracy: 0.9250

-- DT --
DT best params: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
DT cross-validation accuracy: 0.9357
DT test set accuracy: 0.9429
DT best max_depth: 3
DT best min_samples_split: 2
DT best min_samples_leaf: 1
DT cross-validation accuracy: 0.9357
DT test set accuracy: 0.9429

-- Ada --
Ada best params: {'learning_rate': 0.2, 'n_estimators': 150}
Ada cross-validation accuracy: 0.9455
Ada test set accuracy: 0.9429
Ada best n_estimators: 150
Ada best learning_rate: 0.2
Ada cross-validation accuracy: 0.9455
Ada test set accuracy: 0.9429

-- GB --
GB best params: {'learning_rate': 0.1, 'max_depth': 1, 'n_estimators': 50}
GB cross-validation accuracy: 0.9446
GB test set accuracy: 0.9429
GB best max_depth: 1
GB best n_estimators: 50
GB best learning_rate:

### Test your code

In [56]:
# --- Runnability: full Part 1 + Part 2 on test-before.csv (not for reported results) ---

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
    GridSearchCV,
    cross_val_score,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.svm import SVC

DATA_PATH = "test-before.csv"

# --- same preprocessing idea as rice (last column = label) ---
df = pd.read_csv(DATA_PATH)
df = df.replace(["?", "NA", "N/A", "na", ""], np.nan)

target_col = df.columns[-1]
X_df = df.drop(columns=[target_col])
y_sr = df[target_col]

imp = SimpleImputer(missing_values=np.nan, strategy="mean")
x = imp.fit_transform(X_df)

scaler = MinMaxScaler()
x = scaler.fit_transform(x)

y = y_sr.replace({"class1": 0, "class2": 1})

# --- Part 1 ---
cvKFold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

log_reg = LogisticRegression(random_state=0)
lr_scores = cross_val_score(log_reg, x, y, cv=cvKFold)
print("test-before | LogR CV mean:", f"{lr_scores.mean():.4f}")

nb = GaussianNB()
nb_scores = cross_val_score(nb, x, y, cv=cvKFold)
print("test-before | NB CV mean:", f"{nb_scores.mean():.4f}")

# --- Part 2 ---
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=0
)

knn_param_grid = {"n_neighbors": [1, 3, 5, 7], "p": [1, 2]}
dt_param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
ada_param_grid = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.1, 0.2, 0.3, 0.5],
}
gb_param_grid = {
    "max_depth": [1, 3, 5, 7],
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.1, 0.2, 0.3, 0.5],
}
rf_param_grid = {"n_estimators": [10, 30, 60, 100], "max_leaf_nodes": [6, 12]}
svm_param_grid = {
    "C": [0.01, 0.1, 1, 5],
    "gamma": [0.01, 0.1, 1, 10],
    "kernel": ["rbf"],
}

def run_grid_search(name, estimator, param_grid):
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cvKFold,
        scoring="accuracy",
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    pred = grid.best_estimator_.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"test-before | {name} best params:", grid.best_params_)
    print(f"test-before | {name} CV acc:", f"{grid.best_score_:.4f}", "| test acc:", f"{acc:.4f}")
    return grid, pred

run_grid_search("KNN", KNeighborsClassifier(), knn_param_grid)
run_grid_search("DT", DecisionTreeClassifier(random_state=0), dt_param_grid)
run_grid_search("Ada", AdaBoostClassifier(random_state=0), ada_param_grid)
run_grid_search("GB", GradientBoostingClassifier(random_state=0), gb_param_grid)

rf_grid, rf_pred = run_grid_search(
    "RF",
    RandomForestClassifier(
        criterion="entropy", max_features="sqrt", random_state=0
    ),
    rf_param_grid,
)
print(
    "test-before | RF macro F1:",
    f"{f1_score(y_test, rf_pred, average='macro'):.4f}",
    "| weighted F1:",
    f"{f1_score(y_test, rf_pred, average='weighted'):.4f}",
)

run_grid_search("SVM", SVC(random_state=0), svm_param_grid)

print("test-before | full Part1/Part2 run finished OK.")

test-before | LogR CV mean: 0.6700
test-before | NB CV mean: 0.6555
test-before | KNN best params: {'n_neighbors': 3, 'p': 1}
test-before | KNN CV acc: 0.7180 | test acc: 0.5476
test-before | DT best params: {'max_depth': 7, 'min_samples_leaf': 4, 'min_samples_split': 10}
test-before | DT CV acc: 0.7835 | test acc: 0.6667
test-before | Ada best params: {'learning_rate': 0.5, 'n_estimators': 150}
test-before | Ada CV acc: 0.8077 | test acc: 0.6667
test-before | GB best params: {'learning_rate': 0.3, 'max_depth': 1, 'n_estimators': 100}
test-before | GB CV acc: 0.8199 | test acc: 0.7143
test-before | RF best params: {'max_leaf_nodes': 12, 'n_estimators': 60}
test-before | RF CV acc: 0.8202 | test acc: 0.6667
test-before | RF macro F1: 0.6541 | weighted F1: 0.6635
test-before | SVM best params: {'C': 5, 'gamma': 10, 'kernel': 'rbf'}
test-before | SVM CV acc: 0.6993 | test acc: 0.5952
test-before | full Part1/Part2 run finished OK.


## **3. Reflection and Discussion**



3. Reflection and Discussion
Performance of All Classifiers
In this pipeline, we evaluated eight classifiers to distinguish between the cammeo and osmancik rice varieties. Part 1 established our baselines using 10-fold stratified cross-validation without hyperparameter tuning, where Logistic Regression achieved a cross-validation (CV) accuracy of 93.86% and Naïve Bayes achieved 92.64%. The strong performance of the linear baseline suggests the dataset features are highly discriminative, though the slightly lower Naïve Bayes score indicates the assumption of feature independence may not perfectly hold for morphological grain traits.

In Part 2, hyperparameter-tuned models generally outperformed the baselines. The tree-based ensemble methods proved to be the most robust. AdaBoost achieved the highest overall CV accuracy at 94.55%, closely followed by Gradient Boosting at 94.46%. When evaluated on the isolated 20% test set, AdaBoost, Gradient Boosting, Random Forest, and a single Decision Tree all converged on an impressive test accuracy of 94.29%. For the Random Forest classifier specifically, this yielded a macro average F1 score of 0.9414 and a weighted average F1 score of 0.9427, demonstrating that the model accurately balanced its predictions across both rice classes without bias. K-Nearest Neighbours yielded the lowest test accuracy of the tuned models (92.50%), which is a common limitation of distance-based algorithms in multidimensional feature spaces, even after Min-Max normalization.

The Impact of Hyperparameter Tuning
Hyperparameter tuning via GridSearchCV combined with 10-fold stratified cross-validation was critical for optimizing model capacity and preventing overfitting to the training data.

Instead of relying on default values, the grid search systematically identified configurations that generalized best to unseen data:

Constraining Complexity: For the Decision Tree, the grid search selected a max_depth of 3 and min_samples_split of 2. By aggressively constraining the tree, the tuning prevented the model from memorizing noise in the training set, allowing a single, simple tree to generalize well enough to match the complex ensemble models on the test set (94.29%).

Balancing Learning Rate and Estimators: For the boosting algorithms, tuning perfectly balanced the trade-off between learning speed and model complexity. Gradient Boosting achieved optimal performance by utilizing 50 estimators but restricting them to very shallow "decision stumps" (max_depth: 1) with a learning_rate of 0.1, ensuring a slow, stable learning trajectory.

Optimizing Margins: For the Support Vector Machine, tuning identified an optimal C value of 5 and gamma of 1 with an rbf kernel. This configuration struck a necessary balance, applying a moderate penalty for misclassifications while tightening the sphere of influence for support vectors to properly capture the non-linear boundaries of the morphological features.

Finally, hyperparameter tuning successfully navigated the bias-variance trade-off, ensuring the final models were sufficiently complex to separate the classes, but sufficiently constrained to perform reliably on the unseen test set.
